# Desigualdades socioeconómicas y salud en España — ESdE 2023

Análisis reproducible de la salud autopercibida según educación, clase social e ingresos. El notebook funciona como recorrido principal del proyecto; la lógica auditable se conserva en `scripts/` y las decisiones metodológicas en `docs/project/`.

## 1. Pregunta e hipótesis

**Pregunta principal:** ¿se mantiene el gradiente socioeconómico en salud autopercibida después de ajustar por edad, sexo, país de nacimiento y comunidad autónoma?

**Hipótesis:** la prevalencia de salud regular, mala o muy mala será mayor en los grupos con menor educación, clase social e ingresos, incluso después del ajuste. El diseño es transversal: se estiman asociaciones, no efectos causales.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
PROCESSED = ROOT / 'data/processed/esde2023_adultos_analitica.csv.gz'
print(f'Proyecto: {ROOT.name}')
print(f'Tabla analítica disponible: {PROCESSED.exists()}')

## 2. Datos oficiales

Los microdatos no se alojan en GitHub. Descarga los paquetes CSV de Adulto y Hogar desde el banco de microdatos del Ministerio de Sanidad y colócalos siguiendo `data/README.md`. Se emplearon el fichero adulto vigente tras la revisión del IMC de 01/09/2025 y el fichero de hogar vigente tras la actualización de ingresos de 16/04/2026.

## 3. Limpieza, unión y validación

La siguiente celda construye la tabla analítica sin modificar los originales, documenta exclusiones y genera controles de calidad y un informe de valores perdidos.

In [ ]:
%run scripts/clean_esde2023.py

In [ ]:
df = pd.read_csv(PROCESSED, low_memory=False)
print(f'{df.shape[0]:,} filas × {df.shape[1]} columnas')
df[['salud_menos_que_buena', 'educacion_3', 'clase_social_1_6', 'ingresos_5', 'FACTORADULTO']].head()

## 4. EDA ponderada y valores perdidos

Se describen prevalencias, distribuciones sociales y selectividad de la información ausente antes de ajustar modelos.

In [ ]:
%run scripts/eda_ponderada.py

In [ ]:
pd.read_csv('reports/eda/indicadores_ponderados_generales.csv')

## 5. Varianzas e intervalos de confianza

El fichero público incorpora estratos, pero no identifica unidades primarias ni aporta pesos replicados. Por ello se usa bootstrap por estrato público como aproximación; los intervalos no equivalen al jackknife oficial por sección censal.

In [ ]:
%run scripts/estimar_varianzas_ic.py

## 6. Contrastes ajustados y regresiones ponderadas

Se ajustan modelos separados por exposición. M0 es bruto; M1 incorpora edad y sexo; M2 añade nacimiento y comunidad autónoma; M3 añade actividad física; y M4, apoyo social. La estandarización marginal traduce los modelos a prevalencias, diferencias y razones ajustadas.

In [ ]:
%run scripts/regresiones_ponderadas.py

In [ ]:
ajustados = pd.read_csv('reports/modelos/contrastes_ajustados_m2.csv')
ajustados

## 7. Conclusiones

La salud menos que buena afecta al 29,1 % de la población ponderada. En el modelo principal, educación baja frente a alta presenta una diferencia de 9,7 puntos porcentuales (RP 1,46); clase VI frente a I, 12,2 puntos (RP 1,57); e ingresos más bajos frente a más altos, 9,6 puntos (RP 1,39). Los gradientes persisten tras el ajuste, pero no deben interpretarse causalmente.

Consulta `reports/executive_summary.md` para la síntesis y `reports/modelos/informe_regresiones_ponderadas.md` para el detalle completo.